In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\yelle\Downloads\sih26\data\0_master_work_lifecycle.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (54576, 55)

Columns:
['Work ID', 'Work Title', 'Lifecycle Stage', 'State', 'Work Category', 'IDA', 'MP Name', 'MP Term', 'Chamber', 'Work Description', 'Rec_Recommended date', 'Rec_Recommended Amount (Rs)', 'San_Sanction Date', 'San_Sanction Amount (Rs)', 'San_Work Status', 'Comp_Completion Date', 'Comp_Amount Disbursed (Rs)', 'Total_Fund_Disbursed', 'Num_Payments', 'Latest_Payment_Status', 'Days Recommended to Sanctioned', 'Days Sanctioned to Completed', 'Sanction vs Recommended Amount Diff', 'Disbursed vs Sanctioned Amount Diff', 'Disbursed vs Total Payments Diff', 'Rec_Work Category', 'Rec_State', 'Rec_IDA', 'Rec_Constituency', 'Rec_Work Description', 'Rec_Sanction Date', 'Rec_Chamber', 'Rec_Work Title (from ID field)', 'Rec_MP Name', 'Rec_MP Term', 'San_Work Category', 'San_State', 'San_IDA', 'San_Constituency', 'San_Work Description', 'San_Recommended date', 'San_Chamber', 'San_Work Title (from ID field)', 'San_MP Name', 'San_MP Term', 'Comp_Work Category', 'Comp_State', '

,Work ID,Work Title,Lifecycle Stage,State,Work Category,IDA,MP Name,MP Term,Chamber,Work Description,...,Comp_Work Category,Comp_State,Comp_IDA,Comp_Work Description,Comp_Constituency,Comp_Image,Comp_Chamber,Comp_Work Title (from ID field),Comp_MP Name,Comp_MP Term
0,WS/MP001/2023-2024/103702,Installation of multi-gym equipment,Completed,Bihar,Normal/Others,SAMASTIPUR(DISTRICT PLANNING OFFICER SAMASTIPU...,Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),Installation Work of Gym Equipment In Governm...,...,Normal/Others,Bihar,SAMASTIPUR(DISTRICT PLANNING OFFICER SAMASTIPU...,Installation Work of Gym Equipment In Governm...,NaN,Images,Rajya Sabha (Elected MP),Installation of multi-gym equipment,Shri Shambhu Sharan Patel,2022-28
1,WS/MP001/2023-2024/1215,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),P.C.C ??? ?? ???????,...,Normal/Others,Bihar,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),P.C.C ??? ?? ???????,NaN,NaN,Rajya Sabha (Elected MP),"Construction of roads, link roads, pathways or...",Shri Shambhu Sharan Patel,2022-28
2,WS/MP001/2023-2024/1216,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),P.C.C ??? ?? ???????,...,Normal/Others,Bihar,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),P.C.C ??? ?? ???????,NaN,NaN,Rajya Sabha (Elected MP),"Construction of roads, link roads, pathways or...",Shri Shambhu Sharan Patel,2022-28
3,WS/MP001/2023-2024/121691,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),Construction work of underground drain and PCC...,...,Normal/Others,Bihar,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Construction work of underground drain and PCC...,NaN,Images,Rajya Sabha (Elected MP),"Construction of roads, link roads, pathways or...",Shri Shambhu Sharan Patel,2022-28
4,WS/MP001/2023-2024/1217,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),P.C.C ??? ?? ???????,...,Normal/Others,Bihar,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),P.C.C ??? ?? ???????,NaN,NaN,Rajya Sabha (Elected MP),"Construction of roads, link roads, pathways or...",Shri Shambhu Sharan Patel,2022-28


In [14]:
# 1. Keep only the columns that actually matter — drop redundant Rec_/San_/Comp_ duplicates
keep_cols = [
    'Work ID', 'Work Title', 'Lifecycle Stage', 'State', 'Work Category', 'IDA',
    'MP Name', 'MP Term', 'Chamber', 'Work Description',
    'Rec_Recommended date', 'Rec_Recommended Amount (Rs)',
    'San_Sanction Date', 'San_Sanction Amount (Rs)', 'San_Work Status',
    'Comp_Completion Date', 'Comp_Amount Disbursed (Rs)',
    'Total_Fund_Disbursed', 'Num_Payments', 'Latest_Payment_Status',
    'Days Recommended to Sanctioned', 'Days Sanctioned to Completed',
    'Sanction vs Recommended Amount Diff', 'Disbursed vs Sanctioned Amount Diff',
    'Disbursed vs Total Payments Diff'
]
df = df[keep_cols].copy()

# 2. Bring back Constituency (only exists under prefixed versions)
raw = pd.read_csv(r"C:\Users\yelle\Downloads\sih26\data\0_master_work_lifecycle.csv")
df['Constituency'] = raw['Rec_Constituency'].combine_first(raw['San_Constituency']).combine_first(raw['Comp_Constituency'])

# 3. Clean garbage placeholder values (like "PC.C?????") across all text columns
garbage_pattern = r'^\s*$|[\?]{2,}|^N/?A$|^NULL$|^-$'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(garbage_pattern, np.nan, regex=True)

# 4. Parse dates properly
for col in ['Rec_Recommended date', 'San_Sanction Date', 'Comp_Completion Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 5. Clean numeric/amount columns (strip commas, currency symbols if any)
amount_cols = [
    'Rec_Recommended Amount (Rs)', 'San_Sanction Amount (Rs)', 'Comp_Amount Disbursed (Rs)',
    'Total_Fund_Disbursed', 'Sanction vs Recommended Amount Diff',
    'Disbursed vs Sanctioned Amount Diff', 'Disbursed vs Total Payments Diff'
]
for col in amount_cols:
    df[col] = df[col].astype(str).str.replace(r'[₹,Rs\s]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.shape)
print(df.isnull().sum().sort_values(ascending=False).head(10))
df.head()

C:\Users\yelle\AppData\Local\Temp\ipykernel_2812\4068488270.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


(54576, 26)
Disbursed vs Total Payments Diff       52574
Total_Fund_Disbursed                   50237
Latest_Payment_Status                  50237
Num_Payments                           50237
MP Term                                38669
Disbursed vs Sanctioned Amount Diff    32105
Days Sanctioned to Completed           32073
Days Recommended to Sanctioned         29080
Sanction vs Recommended Amount Diff    29080
Rec_Recommended date                   23824
dtype: int64


,Work ID,Work Title,Lifecycle Stage,State,Work Category,IDA,MP Name,MP Term,Chamber,Work Description,...,Comp_Amount Disbursed (Rs),Total_Fund_Disbursed,Num_Payments,Latest_Payment_Status,Days Recommended to Sanctioned,Days Sanctioned to Completed,Sanction vs Recommended Amount Diff,Disbursed vs Sanctioned Amount Diff,Disbursed vs Total Payments Diff,Constituency
0,WS/MP001/2023-2024/103702,Installation of multi-gym equipment,Completed,Bihar,Normal/Others,SAMASTIPUR(DISTRICT PLANNING OFFICER SAMASTIPU...,Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),Installation Work of Gym Equipment In Governm...,...,1499281.0,NaN,NaN,NaN,33.0,125.0,0.0,0.0,NaN,NaN
1,WS/MP001/2023-2024/1215,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),NaN,...,1484933.0,NaN,NaN,NaN,59.0,42.0,0.0,0.0,NaN,NaN
2,WS/MP001/2023-2024/1216,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),NaN,...,1486089.0,NaN,NaN,NaN,59.0,42.0,0.0,0.0,NaN,NaN
3,WS/MP001/2023-2024/121691,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),Construction work of underground drain and PCC...,...,460800.0,NaN,NaN,NaN,9.0,67.0,0.0,0.0,NaN,NaN
4,WS/MP001/2023-2024/1217,"Construction of roads, link roads, pathways or...",Completed,Bihar,Normal/Others,PATNA(DISTRICT PLANNING OFFICER PATNA_IDA),Shri Shambhu Sharan Patel,2022-28,Rajya Sabha (Elected MP),NaN,...,1484550.0,NaN,NaN,NaN,59.0,42.0,0.0,0.0,NaN,NaN


In [15]:
# Verify duplicate Work IDs
print("Duplicate Work IDs:", df['Work ID'].duplicated().sum())

# Verify NaN pattern lines up with lifecycle stage
print(pd.crosstab(df['Lifecycle Stage'], df['Days Sanctioned to Completed'].isna()))
print(pd.crosstab(df['Lifecycle Stage'], df['Total_Fund_Disbursed'].isna()))

# Fresh look at cleaned data
df.head()
df.describe(include='all')

Duplicate Work IDs: 0
Days Sanctioned to Completed  False  True 
Lifecycle Stage                           
Completed                     22471  18439
Recommended                       0   3109
Sanctioned                       32  10497
Unknown                           0     28
Total_Fund_Disbursed  False  True 
Lifecycle Stage                   
Completed              2002  38908
Recommended             893   2216
Sanctioned             1444   9085
Unknown                   0     28


,Work ID,Work Title,Lifecycle Stage,State,Work Category,IDA,MP Name,MP Term,Chamber,Work Description,...,Comp_Amount Disbursed (Rs),Total_Fund_Disbursed,Num_Payments,Latest_Payment_Status,Days Recommended to Sanctioned,Days Sanctioned to Completed,Sanction vs Recommended Amount Diff,Disbursed vs Sanctioned Amount Diff,Disbursed vs Total Payments Diff,Constituency
count,54576,54576,54576,54576,54569,54576,54576,15907,54576,54302,...,4.091000e+04,4.339000e+03,4339.000000,4339,25496.000000,22503.000000,25496.0,2.247100e+04,2.002000e+03,38669
unique,54576,107,4,34,4,739,668,9,3,48647,...,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,504
top,WS/MP001/2023-2024/103702,"Construction of roads, link roads, pathways or...",Completed,Uttar Pradesh,Normal/Others,JAUNPUR(DISTRICT MAGISTRATE JAUNPUR_IDA),PRIYA SAROJ,2022-28,"Lok Sabha (Elected, Constituency-based)",High Mast LED Light (9.5 mtrs MS Pole with 6 L...,...,NaN,NaN,NaN,Payment Success,NaN,NaN,NaN,NaN,NaN,MACHHLISHAHR(SC)
freq,1,13533,40910,12589,53508,1323,739,8516,38669,204,...,NaN,NaN,NaN,3563,NaN,NaN,NaN,NaN,NaN,739
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.552989e+05,7.078983e+05,1.589306,NaN,122.919399,226.040483,0.0,-3.595318e+03,1.018544e+05,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.448000e+03,1.440000e+02,1.000000,NaN,0.000000,0.000000,0.0,-4.367476e+06,0.000000e+00,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.755998e+05,1.497110e+05,1.000000,NaN,45.000000,102.000000,0.0,0.000000e+00,0.000000e+00,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.000000e+05,4.168310e+05,1.000000,NaN,88.000000,199.000000,0.0,0.000000e+00,0.000000e+00,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.180000e+05,7.867030e+05,1.000000,NaN,160.000000,330.000000,0.0,0.000000e+00,0.000000e+00,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.647040e+07,3.949252e+07,40.000000,NaN,1100.000000,948.000000,0.0,0.000000e+00,1.281216e+07,NaN


In [17]:
completed = df[df['Lifecycle Stage'] == 'Completed'].copy()
in_progress = df[df['Lifecycle Stage'] != 'Completed'].copy()
print("Completed:", len(completed), "| In-progress:", len(in_progress))

Completed: 40910 | In-progress: 13666


In [18]:
completed['timeline_flag'] = completed['Days Sanctioned to Completed'] > 365

cost_features = completed[['Disbursed vs Sanctioned Amount Diff', 'Disbursed vs Total Payments Diff']].fillna(0)
model = IsolationForest(contamination=0.05, random_state=42)
completed['cost_outlier_flag'] = model.fit_predict(cost_features) == -1

completed['payment_mismatch_flag'] = (
    completed['Latest_Payment_Status'].astype(str).str.contains('Paid|Released', case=False, na=False)
    & (completed['San_Work Status'] != 'Completed')
)

print(completed[['timeline_flag','cost_outlier_flag','payment_mismatch_flag']].sum())

timeline_flag            4550
cost_outlier_flag        2045
payment_mismatch_flag       0
dtype: int64


In [19]:
print(completed['Latest_Payment_Status'].value_counts())
print()
print(completed['San_Work Status'].value_counts())

Latest_Payment_Status
Payment Success        1807
Payment In-Progress     195
Name: count, dtype: int64

San_Work Status
Physical Inspection    20388
Work Completed          2083
Name: count, dtype: int64


In [20]:
completed['payment_mismatch_flag'] = (
    (completed['Latest_Payment_Status'].astype(str).str.contains('Success', case=False, na=False))
    & (completed['San_Work Status'] != 'Completed')
)
print(completed['payment_mismatch_flag'].sum())

1807


In [21]:
completed['payment_mismatch_flag'] = (
    (completed['Latest_Payment_Status'].astype(str).str.contains('Success', case=False, na=False))
    & (completed['San_Work Status'] != 'Work Completed')
)
print(completed['payment_mismatch_flag'].sum())
print(completed[['Latest_Payment_Status','San_Work Status']].value_counts())

1669
Latest_Payment_Status  San_Work Status    
Payment Success        Physical Inspection    449
                       Work Completed         138
Payment In-Progress    Physical Inspection     65
                       Work Completed          31
Name: count, dtype: int64


In [22]:
completed['payment_mismatch_flag'] = (
    completed['Latest_Payment_Status'].astype(str).str.contains('Success', case=False, na=False)
    & completed['San_Work Status'].notna()
    & (completed['San_Work Status'] != 'Work Completed')
)
print(completed['payment_mismatch_flag'].sum())

449


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sample = completed.dropna(subset=['Work Description']).sample(min(3000, len(completed)), random_state=42)
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
vectors = tfidf.fit_transform(sample['Work Description'])
sim_matrix = cosine_similarity(vectors)
np.fill_diagonal(sim_matrix, 0)
sample['duplicate_flag'] = sim_matrix.max(axis=1) > 0.85

completed = completed.merge(sample[['Work ID','duplicate_flag']], on='Work ID', how='left')
completed['duplicate_flag'] = completed['duplicate_flag'].fillna(False)
print("Duplicates flagged:", completed['duplicate_flag'].sum())

Duplicates flagged: 1314


In [24]:
# Predictive/early-warning for in-progress works
in_progress['elapsed_days'] = (pd.Timestamp.now() - pd.to_datetime(in_progress['San_Sanction Date'], errors='coerce')).dt.days
in_progress['predicted_overrun_flag'] = in_progress['elapsed_days'] > 365
print("In-progress overrun flagged:", in_progress['predicted_overrun_flag'].sum())

In-progress overrun flagged: 8237


In [28]:
import sys
print("Kernel using:", sys.executable)

!{sys.executable} -m pip install shap

print("Done — now restart the kernel, then run: import shap")

Kernel using: c:\Users\yelle\Downloads\sih26\.venv\Scripts\python.exe



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached shap-0.52.0-cp312-abi3-win_amd64.whl.metadata (26 kB)
     ---------------------------------------- 0.0/57.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/57.3 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.3 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.3 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.3 kB ? eta -:--:--
     ------------- ------------------------- 20.5/57.3 kB 65.6 kB/s eta 0:00:01
     ------------- ------------------------- 20.5/57.3 kB 65.6 kB/s eta 0:00:01
     -------------------- ------------------ 30.7/57.3 kB 93.9 kB/s eta 0:00:01
     -------------------------------------- 57.3/57.3 kB 151.0 kB/s eta 0:00:00
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached numba-0.67.0-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached llvmlite-0.49.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
Using cached shap-0.52.0-cp312-abi3-wi

In [29]:
import shap
explainer = shap.Explainer(model, cost_features)
shap_values = explainer(cost_features)

c:\Users\yelle\Downloads\sih26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Background dataset has 40910 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=40910 when initializing the masker.
100%|===================| 40805/40910 [03:03<00:00]        

In [31]:
feature_cols = cost_features.columns.tolist()

completed['cost_reason'] = [
    f"{feature_cols[np.abs(row).argmax()]} drove the flag"
    for row in shap_values.values
]
print(completed['cost_reason'].value_counts())

cost_reason
Disbursed vs Sanctioned Amount Diff drove the flag    40618
Disbursed vs Total Payments Diff drove the flag         292
Name: count, dtype: int64


In [35]:
completed['payment_mismatch_flag'] = (
    completed['Latest_Payment_Status'].astype(str).str.contains('Success', case=False, na=False)
    & completed['San_Work Status'].notna()
    & (completed['San_Work Status'] != 'Work Completed')
)
print("Payment mismatches flagged:", completed['payment_mismatch_flag'].sum())

Payment mismatches flagged: 449


In [37]:
# ============================================================
# ASSET CREATION EVIDENCE CHECK (rule-based)
# Closest available proxy for "asset creation" in the PS —
# Comp_Image is the record of the physical asset once complete
# ============================================================
completed['Comp_Image'] = raw.loc[completed.index, 'Comp_Image'] if 'Comp_Image' in raw.columns else np.nan

completed['no_asset_evidence_flag'] = (
    completed['San_Work Status'].notna()
    & completed['Comp_Image'].isna()
)
print("Works completed with no asset-creation evidence:", completed['no_asset_evidence_flag'].sum())

Works completed with no asset-creation evidence: 11565


In [38]:
# ============================================================
# REAL PREDICTIVE MODEL — trained on completed works,
# applied to in-progress works to forecast overrun risk
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

predict_features = ['Work Category', 'State', 'San_Sanction Amount (Rs)']
target = 'timeline_flag'  # already 0/1 on completed works

train_data = completed.dropna(subset=predict_features + [target])

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['Work Category', 'State']),
], remainder='passthrough')

overrun_model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced'))
])

overrun_model.fit(train_data[predict_features], train_data[target])

# Apply to in-progress works — this is the actual "predictive insight"
predict_data = in_progress.dropna(subset=predict_features).copy()
predict_data['predicted_overrun_prob'] = overrun_model.predict_proba(predict_data[predict_features])[:, 1]

in_progress = in_progress.merge(
    predict_data[['Work ID', 'predicted_overrun_prob']], on='Work ID', how='left'
)
in_progress['predicted_overrun_prob'] = in_progress['predicted_overrun_prob'].fillna(0)

# Early warning: flag if model predicts high risk EVEN BEFORE hitting 365 days
in_progress['early_warning_flag'] = (
    (in_progress['predicted_overrun_prob'] > 0.6) & (in_progress['elapsed_days'] < 365)
)
print("High predicted-overrun-risk works:", (in_progress['predicted_overrun_prob'] > 0.6).sum())
print("Early warnings (flagged BEFORE 1-year mark):", in_progress['early_warning_flag'].sum())

High predicted-overrun-risk works: 1529
Early warnings (flagged BEFORE 1-year mark): 80


In [40]:
# ============================================================
# FINAL COMPOSITE RISK SCORE — includes all gap-closing additions
# ============================================================
def build_reason(row):
    reasons = []
    if row.get('timeline_flag'): reasons.append("Exceeded 1-year completion norm")
    if row.get('cost_outlier_flag'): reasons.append(f"Cost outlier — {row.get('cost_reason','')}")
    if row.get('payment_mismatch_flag'): reasons.append("Payment released before work verified complete")
    if row.get('duplicate_flag'): reasons.append("Similar to another work description")
    if row.get('no_asset_evidence_flag'): reasons.append("Marked complete with no asset-creation evidence (no image on record)")
    if row.get('predicted_overrun_flag'): reasons.append("Elapsed time exceeds norm, still in progress")
    if row.get('early_warning_flag'): reasons.append(f"Model predicts overrun risk ({row.get('predicted_overrun_prob',0)*100:.0f}%) before 1-year mark")
    return "; ".join(reasons) if reasons else "No issues detected"

flag_cols_completed = ['timeline_flag', 'cost_outlier_flag', 'payment_mismatch_flag', 'duplicate_flag', 'no_asset_evidence_flag']
completed['risk_score'] = completed[flag_cols_completed].sum(axis=1)
completed['reason'] = completed.apply(build_reason, axis=1)
completed['status_group'] = 'Completed'

in_progress['risk_score'] = in_progress['predicted_overrun_flag'].astype(int) + in_progress['early_warning_flag'].astype(int)
in_progress['reason'] = in_progress.apply(build_reason, axis=1)
in_progress['status_group'] = 'In Progress'

final = pd.concat([completed, in_progress], ignore_index=True)
final['risk_tier'] = pd.cut(final['risk_score'], bins=[-1, 0, 1, 10], labels=['Low', 'Medium', 'High'])

final.to_csv("scored_works.csv", index=False)
print(final['risk_tier'].value_counts())
print("\nSaved:", final.shape)

risk_tier
Low       30938
Medium    19705
High       3933
Name: count, dtype: int64

Saved: (54576, 41)


In [ ]:
print(raw.columns.tolist())

['Work ID', 'Work Title', 'Lifecycle Stage', 'State', 'Work Category', 'IDA', 'MP Name', 'MP Term', 'Chamber', 'Work Description', 'Rec_Recommended date', 'Rec_Recommended Amount (Rs)', 'San_Sanction Date', 'San_Sanction Amount (Rs)', 'San_Work Status', 'Comp_Completion Date', 'Comp_Amount Disbursed (Rs)', 'Total_Fund_Disbursed', 'Num_Payments', 'Latest_Payment_Status', 'Days Recommended to Sanctioned', 'Days Sanctioned to Completed', 'Sanction vs Recommended Amount Diff', 'Disbursed vs Sanctioned Amount Diff', 'Disbursed vs Total Payments Diff', 'Rec_Work Category', 'Rec_State', 'Rec_IDA', 'Rec_Constituency', 'Rec_Work Description', 'Rec_Sanction Date', 'Rec_Chamber', 'Rec_Work Title (from ID field)', 'Rec_MP Name', 'Rec_MP Term', 'San_Work Category', 'San_State', 'San_IDA', 'San_Constituency', 'San_Work Description', 'San_Recommended date', 'San_Chamber', 'San_Work Title (from ID field)', 'San_MP Name', 'San_MP Term', 'Comp_Work Category', 'Comp_State', 'Comp_IDA', 'Comp_Work Descrip